<a href="https://colab.research.google.com/github/Fespinoza1477/walmart-hurricane-project/blob/main/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS 2002 Midterm Project

Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sqlite3 as sql
import datetime as dt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
import re
import requests
import time

Raw Data Files

In [ ]:
# CSV Files
store_locations = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/store_locations.csv'
transactions = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/walmart_transactions.csv'
distribution_center = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/distribution_centers.csv'
product_catalog = 'https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/product_catalog.csv'
# Database File
!wget -O walmart.db "https://raw.githubusercontent.com/Fespinoza1477/walmart-hurricane-project/main/Data/inventory_and_sales.db"
conn = sql.connect("walmart.db")


# Tables from CSV Files
loc = pd.read_csv(store_locations)
trans = pd.read_csv(transactions)
dist = pd.read_csv(distribution_center)
cat = pd.read_csv(product_catalog)

# Tables from database file
inv_levels_db = pd.read_sql("SELECT * FROM inventory_levels;", conn)
daily_sales_db = pd.read_sql("SELECT * FROM daily_sales_summary;", conn)

Cleaning

In [ ]:
# Profile on the tables
def profile(df, name=None):
    """Return a summary of dtypes, null counts, null percentages, and unique counts."""
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })
    if name:
        print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return summary

tables = {"loc": loc, "trans": trans, "dist": dist, "cat": cat, "inv_levels_db":inv_levels_db, "daily_sales_db": daily_sales_db}

for name, df in tables.items():
    display(profile(df, name))
    print()


In [ ]:
# Store Location Cleaning

# Normalizing spelling of Florida to FLORIDA
loc["state"] = (
    loc["state"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(".","", regex=False)
)
state_map = {
    "FLORIDA":"FLORIDA",
    "FLA":"FLORIDA",
    "FL":"FLORIDA",
}
loc["state"]=loc["state"].replace(state_map)

# Fixing store_id to drop duplicates
id_fix = {
    "FL239":"FL-239",
    "FL_287":"FL-287",
}
loc["store_id"]=loc["store_id"].replace(id_fix)
loc = loc.drop_duplicates(subset="store_id", keep="first").reset_index(drop=True)

In [ ]:
# Distribution Cleaning

# Removing "~" and converting "surge_capacity" to Int64
dist["surge_capacity"] = (
    pd.to_numeric(dist["surge_capacity"].astype(str).str.replace("~", "", regex=False), errors="coerce")
    .astype("Int64")
)

# Need to make a decision on what to do for missing cost/shipment for DC-BHM

In [ ]:
# Product Catalog Cleaning and SKU Mapping

# Cleaning category by normalizing case and consolidating category labels
cat["category"] = (
    cat["category"].str.strip().str.lower()
    .str.title()
)

category_map = {
        "Breakfast Foods": "Breakfast",
        "Emergency Supplies": "Emergency",
        "Alcoholic Beverages": "Alcohol",
        "Groceries": "Grocery",
        "Baby": "Baby Products",
        "Games & Entertainment":"Entertainment",
        "Household Essentials":"Household",
        "Health & Wellness":"Health",
}
cat["category"]=cat["category"].replace(category_map)

# SKU mapping. SKUs will be consolidated based on product name, ignoring all other columns such as cost, supplier and package size
sku_map = {
        'PT-12':'POPTART-STRAW',
        'SKU#459812':'POPTART-STRAW',
        'POPTART-STRAW':'POPTART-STRAW',
        'PT_STRAWBERRY':'POPTART-STRAW',
        'WTR-24':'WATER24PK',
        'WATER24PK':'WATER24PK',
        'WTR_24PK':'WATER24PK',
        'BATT-100': 'BAT-AA4',
        'BAT-AA4':'BAT-AA4',
        'BATT_AA_4PK':'BAT-AA4',
        'FLASH-01': 'LED-FLASH',
        'LED-FLASH': 'LED-FLASH',
        'BEER-24': 'BEER-24',
        'LTBEER24': 'BEER-24',
        'BG-MONO': 'GAME-MONO',
        'MONOPOLY-01': 'GAME-MONO',
        'GAME-MONO': 'GAME-MONO',
        'CARDS-01':'CARDS-01',
        'PLAYCARD-STD':'CARDS-01',
        'STREAM-01':'STREAM-01',
        'ROKU-STD':'STREAM-01',
        'SOUP-CAN':'SOUP',
        'CNSOUP-01':'SOUP',
        'BREAD-WH': 'WHITEBREAD',
        'WHITEBREAD': 'WHITEBREAD',
        'PB-16': "PNUTBTR",
        'PNUTBTR-16': "PNUTBTR",
        'CNDL-EM': "EMERG-CNDL",
        'CANDLE-EMRG': "EMERG-CNDL",
        'FAID-01': 'FIRSTAID-KIT',
        'FIRSTAID-KIT': 'FIRSTAID-KIT',
        'CHRG-PORT': 'CHRGR',
        'PORTCHARGER': 'CHRGR',
        'CHIPS-VP': "CHIPS-VP",
        'CHIPVARIETY': "CHIPS-VP",
        'PTOWEL-6': 'PTOWEL',
        'PAPERTOWEL6': 'PTOWEL',
        'TP-12': 'TOILETPAPER',
        'TOILETPAPER12': 'TOILETPAPER',
        'DIAP-S3':'DIAPERS',
        'DIAPERS-3':'DIAPERS',
        'BFORM-01':'BABYFORM',
        'BABYFORMULA':'BABYFORM',
        'TARP-HD': 'TARP-HD',
        'HDTARP-01': 'TARP-HD',
}
cat["sku"]=cat["sku"].replace(sku_map)

# Normalizing product_name. This also has functionality for the transaction data
groups = {
    "Pop-Tarts Strawberry":   ["pop-tarts strawberry", "Strawberry Pop-Tart", 'Pop Tarts Strawberry',"Pop-Tarts Strawberry",'POPTART-STRAWBERRY', 'Pop Tarts - Strawberry'],
    "Bottled Water 24-Pack":  ["BOTTLED WATER 24PK", "Bottled Water 24-Pack", "Water - 24 Pack"],
    "AA Batteries 4-Pack":    ["BATTERIES AA 4PK", "AA Batteries 4-Pack",'AA Batteries (4-pack)'],
    "LED Flashlight":         ["FLASHLIGHT LED", "LED Flashlight",'Flashlight - LED'],
    "Light Beer 24-Pack":     ["Light Beer (24-pack)", "Light Beer 24-Pack",'Light Beer 24pk','BEER LIGHT 24PK'],
    "Roku Streaming Stick":   ["Roku Streaming Stick", "Streaming Device",'STREAMING STICK'],
    "Monopoly Board Game":    ["Monopoly Board Game", "Board Game - Monopoly",'MONOPOLY'],
    "Playing Cards":          ["Playing Cards", "PLAYING CARDS",'Standard Playing Cards'],
    "Canned Soup":            ["Soup - Canned", "Canned Soup",'CANNED SOUP'],
    "White Bread":            ["WHITE BREAD", "White Bread",'Bread - White'],
    "Peanut Butter":          ["PEANUT BUTTER", "Peanut Butter",'Peanut Butter 16oz'],
    "Emergency Candles":      ["Candles (Emergency)", "Emergency Candles",'CANDLES - EMERGENCY'],
    "First Aid Kit":          ["First-Aid Kit", "First Aid Kit",'FIRST AID KIT'],
    "Portable Phone Charger": ["PORTABLE CHARGER", "Portable Phone Charger",'Phone Charger - Portable'],
    "Chips Variety Pack":     ["Variety Pack - Chips", "Chips Variety Pack",'CHIPS VARIETY PACK'],
    "Paper Towels":           ["Paper Towels",  'PAPER TOWELS 6PK','Paper Towels 6-Roll'],
    "Toilet Paper 12-Roll":   ["TOILET PAPER 12PK", "Toilet Paper 12-Roll",'TP 12 Pack'],
    "Baby Diapers - Size 3":  ["Baby Diapers - Size 3", "DIAPERS SZ 3",'Diapers Size 3'],
    "Baby Formula":           ["Infant Formula", "Baby Formula", 'BABY FORMULA'],
    "Heavy Duty Tarp":        ["Heavy Duty Tarp", "TARP - HEAVY DUTY",'HD Tarp'],
}


name_map = {variant: canon for canon, variants in groups.items() for variant in variants}
cat["product_name"] = cat["product_name"].str.strip().map(name_map)

# Converting package_size to Int64. For the entries with <NA>, they were formerly "bundle", which is not clear on how large the package size is.
size = cat["package_size"].str.strip().str.lower().replace({"single": "1", "bundle": pd.NA})
cat["package_size"] = size.str.extract(r"(\d+)", expand=False).astype("Int64")

# Need to decide on what to do with NaN in the "unit_cost" column
#Decision: Left missing product unit costs as NaN. Do not use these rows for cost or margin claims.


In [ ]:
# Transaction data cleaning

trans_raw = trans.copy()

# Dropping duplicates based on transaction ID
trans = trans.drop_duplicates(subset="transaction_id", keep="first").reset_index(drop=True)

# Dropping duplicates based on transaction ID
#Removed trans = trans.drop_duplicates(subset="transaction_id", keep="first").reset_index(drop=True) as it was mentioned in the line before.

# Cleaning category by normalizing case and consolidating category labels
trans["category"] = (
    trans["category"].str.strip().str.lower()
    .str.title()
)
trans["category"]=trans["category"].replace(category_map)

trans["sku_original"] = trans["sku"]

# SKU map
trans["sku"]=trans["sku"].replace(sku_map)


# product names
trans["product_name"] = trans["product_name"].str.strip().map(name_map)
trans.isna().sum()

# store_id
canonical = ['FL-239','FL-105','FL-330','FL-412','FL-287','FL-156',
             'FL-501','FL-078','FL-619','FL-445','FL-302','FL-710']

def clean_code(code):
    m = re.fullmatch(r"FL[-_]?(\d{3})", str(code).strip().upper())
    return f"FL-{m.group(1)}" if m else None

trans["store_id"] = trans["store_id"].map(clean_code)

# converting unit price to float by dropping $
trans["unit_price"] = (
    pd.to_numeric(trans["unit_price"].astype(str).str.replace("$", "", regex=False), errors="coerce")
    .astype("Float64")
)



# converting timestamp to datetime. Michael proposes that we drop rows with NaN for timestamp.
# The rows are useless because they cannot help identify trends. Its just under 900 rows, so thats a lot of data getting axed.
# Debating whether or not to replace the "timestamp" column with simply "date". Both are in datetime fornmat
ts = trans["timestamp"].str.strip().replace(
    {"": None, "NULL": None, "N/A": None, "NaN": None, "nan": None}
)
formats = [
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%dT%H:%M:%S",
    "%m/%d/%Y %H:%M",
    "%d-%b-%Y %H:%M",
    "%m-%d-%Y %H:%M:%S",
]
parsed = pd.Series(pd.NaT, index=trans.index)
for fmt in formats:
    mask = parsed.isna() & ts.notna()
    parsed[mask] = pd.to_datetime(ts[mask], format=fmt, errors="coerce")
#Added single line of code below to actively apply the timezone localization.
parsed = parsed.dt.tz_localize('America/New_York')
#ADDED: Timezone localization required to align with the weather API properly
#Converting local timestamps to Eastern Time (America/New_York)
trans["timestamp"] = parsed

#Extract standardized date
trans["date"] = trans["timestamp"].dt.date
trans['date'] = pd.to_datetime(trans['date'])

# Keep all cleaned transactions, but exclude rows without dates from date-based analysis
analysis_trans = trans.dropna(subset=["date"]).copy()

In [ ]:
# group transaction data and find sales by day
# join weather API with sales/day
# find correlation between weather and sales

In [ ]:
# Cleaning decision log
raw_missing_timestamps = (
    trans_raw["timestamp"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["", "null", "n/a", "nan"])
    .sum()
)

cleaning_log = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "rows affected": trans_raw["transaction_id"].duplicated().sum(),
        "decision": "Kept the first row for each transaction ID."
    },
    {
        "issue": "Missing or unparseable timestamps",
        "rows affected": raw_missing_timestamps,
        "decision": "Kept in trans, excluded from date-based analysis."
    },
    {
        "issue": "Negative quantities",
        "rows affected": (trans["quantity"] < 0).sum(),
        "decision": "Retained as possible returns; they reduce net units sold."
    },
    {
        "issue": "Missing product unit costs",
        "rows affected": cat["unit_cost"].isna().sum(),
        "decision": "Left missing; do not use these rows for cost or margin claims."
    },
    {
        "issue": "Missing distribution-center shipping costs",
        "rows affected": dist["cost_per_shipment"].isna().sum(),
        "decision": "Left missing; do not compare shipping costs for that center."
    }
])

display(cleaning_log)
print("Rows in raw transactions:", len(trans_raw))
print("Rows after duplicate removal:", len(trans))
print("Rows available for date-based analysis:", len(analysis_trans))

###Q1: Demand Surge - Clean timestamps and categories, a baseline window you can defend, and a chart of the top 5 categories

In [ ]:
trans.head()

In [ ]:
# Visualization of Top 5 Categories Sales over Time
daily = (trans.groupby(["date", "category"])["quantity"].sum()
              .unstack("category").fillna(0))

top5 = daily.sum().nlargest(5).index

fig, ax = plt.subplots(figsize=(12, 6))
daily[top5].plot(ax=ax, x_compat=True, marker="o", markersize=4)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=45, ha="right")

ax.set_title("Daily Units Sold by Category (Top 5)")
ax.set_xlabel("Date")
ax.set_ylabel("Units per Day")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Define the missing variables
baseline_start = pd.Timestamp("2024-09-01")
baseline_end = pd.Timestamp("2024-09-15")
surge_start = pd.Timestamp("2024-09-23")
surge_end = pd.Timestamp("2024-09-25")
landfall = pd.Timestamp("2024-09-26")

# Visualization of Top 5 Categories Sales over Time
daily = (trans.groupby(["date", "category"])["quantity"].sum()
              .unstack("category").fillna(0))

top5 = daily.sum().nlargest(5).index

fig, ax = plt.subplots(figsize=(12, 6))
daily[top5].plot(ax=ax, x_compat=True, marker="o", markersize=4)

# Baseline and surge windows
ax.axvspan(baseline_start, baseline_end, color="gray", alpha=0.1, label="Baseline Window")
ax.axvspan(surge_start, surge_end, color="orange", alpha=0.1, label="Surge Window")
ax.axvline(landfall, color="red", linestyle="--", label="Landfall: Sept. 26")

ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=45, ha="right")

ax.set_title("Daily Units Sold by Category (Top 5)")
ax.set_xlabel("Date")
ax.set_ylabel("Units per Day")
ax.legend(title="Category")
plt.tight_layout()
plt.show()

In [ ]:
# Visualization Option 2
# Q1: Compare daily units sold with a pre-storm baseline
date_range = pd.date_range("2024-09-01", "2024-09-30")

daily = (
    analysis_trans.groupby(["date", "category"])["quantity"]
    .sum()
    .unstack("category")
    .reindex(date_range, fill_value=0)
)

top5 = daily.sum().nlargest(5).index

baseline_start = pd.Timestamp("2024-09-01") #Edited: Expanded baseline to start on Sep 1 for a more robust 15-day average
baseline_end = pd.Timestamp("2024-09-15")
surge_start = pd.Timestamp("2024-09-23")
surge_end = pd.Timestamp("2024-09-25")
landfall = pd.Timestamp("2024-09-26")

baseline_mean = daily.loc[baseline_start:baseline_end, top5].mean()
surge_mean = daily.loc[surge_start:surge_end, top5].mean()

q1_results = pd.DataFrame({
    "baseline_units_per_day": baseline_mean,
    "surge_units_per_day": surge_mean
})
# Calculate growth metrics
q1_results["surge_multiple"] = (
    q1_results["surge_units_per_day"] /
    q1_results["baseline_units_per_day"]
)
q1_results["percent_change"] = (
    (q1_results["surge_units_per_day"] /
     q1_results["baseline_units_per_day"] - 1) * 100
)
print("=== Q1 Numerical Findings ===")
display(q1_results.sort_values("surge_multiple", ascending=False).round(2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

# Plot 1: daily units sold
daily[top5].plot(ax=ax1, marker="o", markersize=4, linewidth=2, x_compat=True)
ax1.axvline(landfall, color="red", linestyle="--", label="Landfall: Sept. 26")

# Shaded windows (added before the legend so they show up in it)
ax1.axvspan(baseline_start, baseline_end, color="gray", alpha=0.1, label="Baseline Window")
ax1.axvspan(surge_start, surge_end, color="orange", alpha=0.1, label="Surge Window")

ax1.set_title("Daily Units Sold in the Five Largest Categories")
ax1.set_xlabel("Date")
ax1.set_ylabel("Units sold per day")

# Label every 2nd tick only
ax1.set_xticks(date_range[::2])
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax1.tick_params(axis="x", rotation=90)
ax1.legend(title="Category")

# Plot 2: bar chart of percent change
q1_plot = q1_results.sort_values("percent_change")
q1_plot["percent_change"].plot(kind="barh", ax=ax2, color="steelblue")
ax2.axvline(0, color="black", linewidth=1)
ax2.set_title("Percent Change: Surge vs. Baseline")
ax2.set_xlabel("Change from baseline (%)")
ax2.set_ylabel("Category")

plt.tight_layout()
plt.show()

Answer: The data confirms a massive demand surge in the days immediately preceding the hurricane's landfall on September 26,2024. Breakfast foods experienced the most dramatic relative spike soaring over 620% from a baseline of ~73 units per day to over 524 units per day during the surge window. Emergency supplied saw the largest absolute volume increase, jumping from a baseline of ~351 units to a staggering 1,335 units per day (a 280% increase). Entertainment, Household, and Grocery categories also saw a significant, though comparatively smaller, sugres ranging from 115% to 196% above normal levels. This suggests consumsers heavily prioritized immediate survival and non-perishable food prep (like Pop-Tarts and bottled water) ahead of general groceries as the storm approached.

In [ ]:
# Visualization option 2
# Q1: Show how each category changed from baseline to pre-landfall surge
q1_plot = q1_results.sort_values("percent_change")

fig, ax = plt.subplots(figsize=(9, 5))
q1_plot["percent_change"].plot(kind="barh", ax=ax, color="steelblue")

ax.axvline(0, color="black", linewidth=1)
ax.set_title("Change in Daily Units Sold Before Landfall")
ax.set_xlabel("Change from baseline (%)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare daily data in long format for Seaborn
daily.index.name = "date"
daily_long = daily[top5].reset_index().melt(id_vars='date', var_name='category', value_name='units_sold')

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=daily_long, x='date', y='units_sold', hue='category', marker='o', ax=ax)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45, ha='right')

ax.set_title('Daily Units Sold by Category (Top 5) - Seaborn')
ax.set_xlabel('Date')
ax.set_ylabel('Units per Day')
plt.tight_layout()
plt.show()

###Q2: SKU Consolidation - Done in the above cleaning

In [ ]:
# Q2: Count how many original SKUs were combined into each cleaned SKU
sku_comparison = (
    trans.groupby(["sku", "product_name"], as_index=False)
    .agg(
        original_sku_count=("sku_original", "nunique"),
        units_sold=("quantity", "sum"),
        transactions=("transaction_id", "nunique")
    )
    .sort_values("original_sku_count", ascending=False)
)

print("Distinct original SKUs:", trans["sku_original"].nunique())
print("Distinct cleaned SKUs:", trans["sku"].nunique())
display(sku_comparison.head(15))

# Plot only cleaned products that combined multiple original SKUs
sku_plot = sku_comparison[sku_comparison["original_sku_count"] > 1].head(12)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(sku_plot["product_name"], sku_plot["original_sku_count"], color="steelblue")
ax.xaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

ax.set_title("Raw SKU Variants for Each Product")
ax.set_xlabel("Number of Raw SKUs")
ax.set_ylabel("Product")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
#Q2: SKU consolidation
#Which products are fragmented, and what changes when you fix it?

#different SKUs were hiding under the same actual product.
trans_raw_mapped = trans_raw.copy()
trans_raw_mapped['canonical_name'] = trans_raw_mapped['product_name'].str.strip().map(name_map)

#ADDED: Group by the real product name and count unique SKUs to find the worst offenders.
fragmentation = trans_raw_mapped.groupby('canonical_name')['sku'].nunique().sort_values(ascending=False)
fragmented_items = fragmentation[fragmentation > 1]

print("=== Most Fragmented Products (Before Cleaning) ===")
display(pd.DataFrame({'Unique SKUs' : fragmented_items}))

#ADDED: Create a before-and-after summary table to satisfy the project brief's requirement
q2_summary = pd.DataFrame({
    'Metric': ['Distinct SKUs', 'Distinct Product Names'],
    'Before Consolidation (Raw)': [trans_raw['sku'].nunique(), trans_raw['product_name'].nunique()],
    'After Consolidation (Clean)': [trans['sku'].nunique(), trans['product_name'].nunique()]
})

print("\n=== Before and After SKU Consolidation ===")
display(q2_summary)

Before consolidation, the dataset presented a fasely inflated catalog consisting of 63 distinct product names and 45 distinct SKUs. This was driven by severe fragmentation, with high-demand items like Pop-Tarts, 12-roll Toilet Paper, and AA Batteries spread across up to 4 different SKUs and spelling variations each. By standardizing the name conventions and consolidating SKUs, we successfully reduced the data to exactly 20 canonical products and 20 distinct SKUs, creating a clean one-to-one mapping. Fixing this fragmentation was critical; without it, our volume analysis would have drastically underreported the true demand for these essential supplies by splitting sales across duplicate records.

###Q3: Weather Correlation - The API call, timezone alignment, a checked join, then correlation and lag

In [ ]:
# Load in API
BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame

# A saved copy of the same Open-Meteo data (Orlando, Sept 1-30, 2024) for when you have no wifi
# or the API is down. The source column says so.
FALLBACK = pd.DataFrame({
    'date': pd.date_range('2024-09-01', periods=30),
    'precipitation_sum': [3.0, 12.4, 24.9, 22.1, 23.1, 9.3, 6.8, 94.7, 28.0, 14.4,
                          12.0, 0.7, 0.6, 21.1, 1.2, 7.9, 2.3, 13.3, 1.4, 0.7,
                          0.4, 2.2, 1.3, 0.3, 6.4, 15.2, 2.5, 18.0, 4.6, 3.3],
    'windspeed_10m_max': [18.8, 13.5, 22.8, 15.5, 15.2, 9.0, 14.2, 12.6, 11.5, 14.3,
                          15.8, 12.1, 10.4, 17.4, 9.0, 13.3, 12.4, 9.3, 12.9, 15.3,
                          19.9, 17.9, 16.6, 17.7, 27.2, 48.8, 45.1, 19.8, 11.7, 8.6],
    'source': 'fallback sample',
})

try:
    weather_live = daily_weather('2024-09-01', '2024-09-30')
except requests.RequestException as err:
    print('API call failed:', err)
    weather_live = FALLBACK

same = (np.allclose(weather_live['precipitation_sum'], FALLBACK['precipitation_sum'])
        and np.allclose(weather_live['windspeed_10m_max'], FALLBACK['windspeed_10m_max']))
print('source:', weather_live['source'].iloc[0])
print('matches the saved copy:', same)

# Creating a sales table that only includes data on total sales per day
sales = analysis_trans.groupby('date')['quantity'].sum().reset_index()
sales = sales.rename(columns={'quantity': 'sales'})

# Adding in landfall date
LANDFALL = pd.Timestamp('2024-09-26')

# API Join and Merge
weather = weather_live        # works whether it came from the API or the saved copy

merged = sales.merge(weather, on='date', how='left', indicator=True)

assert len(merged) == len(sales), 'the join changed the row count'
assert not (merged['_merge'] == 'right_only').any(), 'weather rows with no sales day'   # a left join can't make these; this guards against someone changing how= later
assert merged['sales'].sum() == sales['sales'].sum(), 'the sales total changed'

outer = sales.merge(weather, on='date', how='outer', indicator=True)

print(merged['_merge'].value_counts())
print('weather days with no sales, dropped on purpose by the left join:', len(weather) - len(sales))
print('an outer join would have added', (outer['_merge'] == 'right_only').sum(), 'right_only rows with no sales')
print('all three checks passed')

# Getting correlation/surge/baseline data
def window(df, start, end):
    return df[(df['date'] >= start) & (df['date'] <= end)]

baseline = window(merged, '2024-09-08', '2024-09-15')    # flat, before the first rise
surge = window(merged, '2024-09-23', '2024-09-25')       # the three days before landfall

baseline_per_day = baseline['sales'].mean()
surge_per_day = surge['sales'].mean()
multiple = surge_per_day / baseline_per_day

print('baseline:', round(baseline_per_day, 1), 'orders/day over', len(baseline), 'days')
print('surge:   ', round(surge_per_day, 1), 'orders/day over', len(surge), 'days')
print('multiple:', round(multiple, 1), 'x')

print()
print('Same data, different windows:')
choices = {
    'baseline Sept 8-15, surge Sept 23-25 (ours)': ('2024-09-08', '2024-09-15', '2024-09-23', '2024-09-25'),
    'baseline Sept 8-22 (runs through the climb)': ('2024-09-08', '2024-09-22', '2024-09-23', '2024-09-25'),
    'surge Sept 24-26 (includes landfall day)':    ('2024-09-08', '2024-09-15', '2024-09-24', '2024-09-26'),
}
for name, (b0, b1, s0, s1) in choices.items():
    m = window(merged, s0, s1)['sales'].mean() / window(merged, b0, b1)['sales'].mean()
    print(f'{m:.1f}x  {name}')


# Lag purchasing
for k in range(0, 5):
    r = merged['sales'].corr(merged['windspeed_10m_max'].shift(-k))
    print(f'orders vs wind {k} day(s) later:  r = {r:.2f}')

# Sales and Windspeed Graph
fig, ax = plt.subplots(figsize=(10, 4.5))

ax.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

ax.plot(merged['date'], merged['sales'], marker='o', color='#24323E', label='Orders per day')
ax.axvline(LANDFALL, color='#F67132', linestyle='--', label='Landfall (Sept 26)')
ax.set_xlabel('Date')
ax.set_ylabel('Orders per day')

ax2 = ax.twinx()
ax2.plot(merged['date'], merged['windspeed_10m_max'], color='#21688D', alpha=0.8, label='Max wind (km/h)')
ax2.set_ylabel('Max wind speed (km/h)')

ax.set_title(f'Orders ran {multiple:.1f}x normal before landfall, about two days ahead of the wind')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left')
fig.autofmt_xdate()
plt.show()

### Weather and Sales Correlation Analysis
We will now calculate the correlation coefficients between daily sales and our daily weather metrics (`precipitation_sum` and `windspeed_10m_max`). Additionally, we analyze lag correlations to understand how many days in advance customers react to the incoming storm relative to the actual weather peak.

In [ ]:
# Calculate contemporaneous correlation
precip_corr = merged['sales'].corr(merged['precipitation_sum'])
wind_corr = merged['sales'].corr(merged['windspeed_10m_max'])

print(f"Contemporaneous correlation (Day 0):")
print(f"  - Sales vs Precipitation: r = {precip_corr:.2f}")
print(f"  - Sales vs Max Wind Speed: r = {wind_corr:.2f}")
print("\nLag Analysis (Sales vs Wind Speed shifted by k days):")

# Calculate lag correlations up to 5 days ahead
for k in range(0, 6):
    r_lag = merged['sales'].corr(merged['windspeed_10m_max'].shift(-k))
    print(f"  - Orders vs Wind Speed {k} day(s) later: r = {r_lag:.2f}")

**Ignore the 3 graphs below, just for testing**

In [ ]:
daily_long = daily[top5].reset_index().melt(id_vars='date', var_name='category', value_name='units_sold')

fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=daily_long, x='date', y='units_sold', hue='category', marker='o', ax=ax)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.xticks(rotation=45, ha='right')

ax.set_title('Daily Units Sold by Category (Top 5) - Seaborn')
ax.set_xlabel('Date')
ax.set_ylabel('Units per Day')
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Set the seaborn style
sns.set_theme(style="whitegrid")

fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot daily sales using Seaborn lineplot on the first axis
sns.lineplot(data=merged, x='date', y='sales', ax=ax1, marker='o', color='#24323E', label='Orders per day', linewidth=2)
ax1.axvline(LANDFALL, color='#F67132', linestyle='--', linewidth=2, label='Landfall (Sept 26)')

ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Orders per day', color='#24323E', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#24323E')

# Create second axis for windspeed
ax2 = ax1.twinx()
# Plot windspeed using Seaborn lineplot on the second axis
sns.lineplot(data=merged, x='date', y='windspeed_10m_max', ax=ax2, color='#21688D', alpha=0.8, label='Max wind (km/h)', linewidth=2)

ax2.set_ylabel('Max wind speed (km/h)', color='#21688D', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#21688D')
ax2.grid(False) # Turn off grid for the secondary axis to avoid overlapping lines

# Align legends
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left')
if ax1.get_legend(): ax1.get_legend().remove()
if ax2.get_legend(): ax2.get_legend().remove()
ax1.legend(h1 + h2, l1 + l2, loc='upper left')

plt.title(f'Orders ran {multiple:.1f}x normal before landfall, about two days ahead of the wind', fontsize=12, fontweight='bold', pad=15)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.lineplot(data=merged, x="date", y="sales", marker="o",
             color="#24323E", label="Orders per day", ax=ax)
ax.axvline(LANDFALL, color="#F67132", linestyle="--", label="Landfall (Sept 26)")
ax.set_xlabel("Date")
ax.set_ylabel("Orders per day")

ax2 = ax.twinx()
sns.lineplot(data=merged, x="date", y="windspeed_10m_max",
             color="#21688D", alpha=0.8, label="Max wind (km/h)", ax=ax2)
ax2.set_ylabel("Max wind speed (km/h)")
ax2.grid(False)   # avoid a second set of gridlines on top of the first

ax.set_title(f"Orders ran {multiple:.1f}x normal before landfall, about two days ahead of the wind")

# Merge the two legends into one
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax2.get_legend().remove()
ax.legend(h1 + h2, l1 + l2, loc="upper left")

fig.autofmt_xdate()
plt.show()

In [ ]:
#Q3: Weather Correlation
#Does the storm data actually line up with the sales pattern?
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#1. Weather API Call with Retry Logic
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

#Open-Meteo Archive API for Orlando, FL (Central proxy for Florida)
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    'latitude': 28.5383,
    'longitude': -81.3792,
    'start_date': "2024-09-01",
    'end_date': '2024-09-30',
    'daily': ["precipitation_sum", "wind_speed_10m_max"],
    'timezone': "America/New_York" #Align with the localized sales data
}

response = session.get(url, params=params)
response.raise_for_status()
weather_data = response.json()

#Parse into DataFrame and add the required source column
weather_df = pd.DataFrame(weather_data['daily'])
weather_df['date'] = pd.to_datetime(weather_df['time'])
weather_df['source'] = 'Open-Meteo API'
weather_df = weather_df.drop(columns=['time'])

#2. Checked Join to Sales Data
#Aggregate sales by day first
daily_sales = analysis_trans.groupby('date')['quantity'].sum().reset_index()

print("=== Join Row-Count Check ===")
print(f"Weather data rows: {len(weather_df)}")
print(f"Sales data rows: {len(daily_sales)}")

#Merge weather and sales
weather_sales = pd.merge(daily_sales, weather_df, on='date', how='inner')
print(f"Joined data rows: {len(weather_sales)}")
#3. Correlation Matrix
correlation = weather_sales[['quantity', 'precipitation_sum', 'wind_speed_10m_max']].corr()
print("\n=== Correlation Matrix ===")
display(correlation)

#4. Visualization
fig, ax1 = plt.subplots(figsize=(12,6))

ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_axisbelow(True)
ax2 = ax1.twinx()
ax1.plot(weather_sales['date'], weather_sales['quantity'], 'b-', marker = 'o', label='Total Units Sold')
ax2.plot(weather_sales['date'], weather_sales['wind_speed_10m_max'], 'r--', marker='x', label='Max Wind Speed (km/h)')
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

ax1.set_xlabel('Date')
ax1.set_ylabel('Total Units Sold', color='b')
ax2.set_ylabel('Max Wind Speed (km/h)', color='r')
plt.title('Daily Sales Volume vs Max Wind Speed (September 2024)')
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.85))
plt.show()

The data shows a distinct relationship between the storm's approach and consumer behavior, but it is an anticipatory relationship rather than a direct concurrent correlation. The correlation matrix shows a moderate relationship between overall sales volume and wind speed, but visually plotting them reveals a clear "lag" effect. Sales hit their absolute peak on September 24th and 25th, right before the wind speed reached its maximum intensity on September 26th (Landfall). Once the storm actually hit and wind speeds spiked, sales plummeted to near zero, perfectly illustrating proactive hurrican preparation followed by store closures or residents sheltering in place.

In [ ]:
# Calculate correlations
quantity_wind_corr = weather_sales['quantity'].corr(weather_sales['wind_speed_10m_max'])
quantity_precip_corr = weather_sales['quantity'].corr(weather_sales['precipitation_sum'])

print(f"Correlation between quantity and max windspeed: {quantity_wind_corr:.4f}")
print(f"Correlation between quantity and precipitation: {quantity_precip_corr:.4f}")

print("\n=== Lag Analysis (Sales vs Wind Speed shifted by k days) ===")
for k in range(1,6):
  r_lag = weather_sales['quantity'].corr(weather_sales['wind_speed_10m_max'].shift(-k))
  print(f"Orders vs Wind Speed {k} day(s) later: r = {r_lag:.4f}")

google gemini: This confirms that consumer buying behavior has a moderate contemporaneous relationship with the incoming weather variables. As discussed in your findings, the strong positive correlation with wind speed (which builds up as the storm approaches) and the negative correlation with precipitation reflect that shoppers rush to complete their prep work right before the storm peaks, followed by a sudden freeze in transactions as the physical rain and wind make travel unsafe.




###Q4: Stores and Geography - Store ids cleaned so the store file joins, then a comparison across stores

In [ ]:
# merging latitude and longitude data onto transaction data
# df_main: the frame you want to add lat/long to (e.g. sales or inventory)
# df_geo:  the frame that has store_id, lat, long

lat_long_trans = trans.merge(
    loc[["store_id", "latitude", "longitude"]],   # only bring over the columns you need
    on="store_id",
    how="left",                            # keep every row of df_main
    validate="m:1",                        # many rows in df_main per one row in df_geo
)
display(lat_long_trans.head())

# need category sales by day by store
store_daily_sales = (
    lat_long_trans.dropna(subset=["date"])
    .groupby(["date", "store_id", "category", "latitude", "longitude"], as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "daily_units_sold"})
)
print("\n=== Category Sales by Day by Store ===")
display(store_daily_sales.head())

In [ ]:
weather

In [ ]:
by_store = (
    lat_long_trans.groupby(["date", "store_id", "latitude", "longitude", "category"], as_index=False)
      .agg(num_transactions=("transaction_id", "nunique"),   # how many sales
           units_sold=("quantity", "sum"))                   # total items sold
      .sort_values(["date", "store_id", "category"])
)
display(by_store.head())

# loop the API join for each lat/long in the list. Do that to get distance away from the storm. precipitation data will match the lat/long from the by_store frame.
# return a unique API for each lat/long

# Load in API
BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def daily_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        r.raise_for_status()                       # any other bad status fails loudly
        frame = pd.DataFrame(r.json()['daily'])
        frame = frame.rename(columns={'time': 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame['source'] = 'open-meteo'
        return frame

unique_stores = lat_long_trans.dropna(subset=['latitude', 'longitude'])[['store_id', 'latitude', 'longitude']].drop_duplicates()
store_weather_list = []
print("Fetching localized weather data for each store...")

for _, row in unique_stores.iterrows():
  store = row['store_id']
  lat = row['latitude']
  lon = row['longitude']

  try:
    df_weather = daily_weather('2024-09-01', '2024-09-30', lat=lat, lon=lon)
    df_weather['store_id'] = store
    store_weather_list.append(df_weather)
    time.sleep(0.5)
  except requests.RequestException as err:
    print(f"API call failed for {store} as {lat}, {lon}: {err}")
if store_weather_list:
  all_stores_weather = pd.concat(store_weather_list, ignore_index=True)
  by_store_weather = by_store.merge(
      all_stores_weather[['date', 'store_id', 'precipitation_sum', 'windspeed_10m_max']],
      on = ['date', 'store_id'],
      how='left'
  )
  print("\n=== Store-Level Sales & Localized Weather Merge Complete ===")
  display(by_store_weather.head())
else:
  print("NO weather data was fetched.")

In [ ]:
daily_loc = (
    lat_long_trans.dropna(subset=["date"])
    .groupby(["date", "store_id", "latitude", "longitude", "category"])["quantity"].sum()
    .unstack("category")
    .fillna(0)
    .reset_index()
)

display(daily_loc.head())



In [ ]:
# Q4: Compare store sales changes with local wind exposure
store_points = loc.dropna(subset=["latitude", "longitude"]).copy()

print("Stores excluded because coordinates are missing:")
print(loc.loc[loc["latitude"].isna() | loc["longitude"].isna(), "store_id"].tolist())

weather_frames = []

for row in store_points.itertuples(index=False):
    store_weather = daily_weather(
        "2024-09-01",
        "2024-09-30",
        lat=row.latitude,
        lon=row.longitude
    )
    store_weather["store_id"] = row.store_id
    weather_frames.append(store_weather)
    time.sleep(0.2)

weather_by_store = pd.concat(weather_frames, ignore_index=True)

# Average units per day for the baseline and three pre-landfall days
store_sales = (
    analysis_trans.groupby(["store_id", "date"])["quantity"]
    .sum()
    .rename("units_sold")
    .reset_index()
)

all_dates = pd.date_range("2024-09-01", "2024-09-30")
all_stores = store_points["store_id"].tolist()
full_index = pd.MultiIndex.from_product(
    [all_stores, all_dates],
    names=["store_id", "date"]
)

store_sales = (
    store_sales.set_index(["store_id", "date"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

baseline_store = (
    store_sales[store_sales["date"].between(baseline_start, baseline_end)]
    .groupby("store_id")["units_sold"]
    .mean()
    .rename("baseline_units_per_day")
)

surge_store = (
    store_sales[store_sales["date"].between(surge_start, surge_end)]
    .groupby("store_id")["units_sold"]
    .mean()
    .rename("surge_units_per_day")
)

store_comparison = pd.concat([baseline_store, surge_store], axis=1).reset_index()
store_comparison["percent_change"] = (
    (store_comparison["surge_units_per_day"] /
     store_comparison["baseline_units_per_day"] - 1) * 100
)

# Use each store's highest local wind speed from Sept. 23 through landfall
peak_wind = (
    weather_by_store[
        weather_by_store["date"].between(surge_start, landfall)
    ]
    .groupby("store_id")["windspeed_10m_max"]
    .max()
    .rename("peak_wind_kmh")
    .reset_index()
)

store_comparison = (
    store_comparison
    .merge(store_points[["store_id", "region"]], on="store_id", how="left")
    .merge(peak_wind, on="store_id", how="left")
)

display(store_comparison.sort_values("percent_change", ascending=False))

fig, ax = plt.subplots(figsize=(9, 6))

ax.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
sns.scatterplot(
    data=store_comparison,
    x="peak_wind_kmh",
    y="percent_change",
    hue="region",
    s=100,
    ax=ax
)

for row in store_comparison.itertuples(index=False):
    ax.annotate(row.store_id, (row.peak_wind_kmh, row.percent_change),
                xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_title("Store Sales Change vs. Local Peak Wind")
ax.set_xlabel("Maximum local wind, Sept. 23–26 (km/h)")
ax.set_ylabel("Change in daily units sold from baseline (%)")
plt.tight_layout()
plt.show()

In [ ]:


# Calculate peak/sum local precipitation for each store during the surge window (Sept 23-25)
local_precip = (
    weather_by_store[weather_by_store["date"].between(surge_start, surge_end)]
    .groupby("store_id")["precipitation_sum"]
    .sum()
    .rename("surge_precipitation_sum_mm")
    .reset_index()
)

# Merge local precipitation with the existing store comparison DataFrame
store_comparison_precip = (
    store_comparison.drop(columns=["surge_precipitation_sum_mm"], errors="ignore")
    .merge(local_precip, on="store_id", how="left")
)

display(store_comparison_precip[["store_id", "region", "percent_change", "surge_precipitation_sum_mm"]].sort_values("percent_change", ascending=False))

# Create scatter plot correlating Store Sales Change with Local Surge Precipitation
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=store_comparison_precip,
    x="surge_precipitation_sum_mm",
    y="percent_change",
    hue="region",
    s=100,
    ax=ax
)

for row in store_comparison_precip.itertuples(index=False):
    ax.annotate(row.store_id, (row.surge_precipitation_sum_mm, row.percent_change),
                xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_title("Store Sales Change vs. Local Surge Precipitation (Sept 23–25)")
ax.set_xlabel("Total Local Precipitation (mm)")
ax.set_ylabel("Change in daily units sold from baseline (%)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
#Q4: Store and Geography
#Did stores nearer the storm behave differently?

#Added: Merge the cleaned transaction data with the store locations data frame
#to bring in geographic features like 'region' and 'city'
store_sales = pd.merge(analysis_trans, loc, on='store_id', how='left')

#ADDED: Check for any unmatched stores to ensure the join was successful
unmatched_stores = store_sales[store_sales['region'].isna()]['store_id'].unique()
if len(unmatched_stores) > 0:
    print(f"Warning: The following store IDs did not match: {unmatched_stores}")
else:
    print("Join successful: All transaction store IDs matched to a location.")

#Added: Aggregate daily sales volume by region to compare geographic behavior
geo_sales = store_sales.groupby(['date', 'region'])['quantity'].sum().unstack('region').fillna(0)

#Make sure the index is a true datetime so the matplotlib date locator/formatter work
geo_sales.index = pd.to_datetime(geo_sales.index)

#Restrict to September 1-30, 2024
start_date = pd.Timestamp("2024-09-01")
end_date = pd.Timestamp("2024-09-30")
geo_sales = geo_sales.loc[start_date:end_date]

print("\n=== Pre-Landfall Sales Peak by Region ===")
# Show the peak sales day for each region to see if some surged earlier
peak_sales = geo_sales.idxmax()
peak_volumes = geo_sales.max()
peak_summary = pd.DataFrame({'Peak Date': peak_sales, 'Peak Volume': peak_volumes})
display(peak_summary.sort_values('Peak Date'))

#Added: Visualization of demand by region over time
fig, ax = plt.subplots(figsize=(14, 7))

#Plot each region's trendline (x_compat=True lets pandas use matplotlib's date axis)
geo_sales.plot(ax=ax, linewidth=2, marker='o', markersize=4, x_compat=True)

#Mark the day of landfall
landfall_date = pd.Timestamp("2024-09-26")
ax.axvline(landfall_date, color="red", linestyle="--", linewidth=2, label="Landfall: Sept. 26")

ax.set_title("Daily Units Sold by Store Region (Hurricane Approach)")
ax.set_xlabel("Date")
ax.set_ylabel("Total Units Sold")

#Pad the x-axis by one day on each side so points aren't against the y-axis
ax.set_xlim(start_date - pd.Timedelta(days=1), end_date + pd.Timedelta(days=1))

#Place ticks explicitly every 3 days, with the first label on Sept 1
tick_dates = pd.date_range(start_date, end_date, freq="3D")
ax.set_xticks(mdates.date2num(tick_dates))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

# Move legend outside the plot so it doesn't overlap the data
ax.legend(title='Region', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

# 1. Prepare and clean store weather daily details
weather_by_store_clean = weather_by_store.copy()
weather_by_store_clean['date'] = pd.to_datetime(weather_by_store_clean['date'])

# 2. Merge regional info into transaction details and aggregate sales
store_sales_geo = pd.merge(analysis_trans, loc[['store_id', 'region']], on='store_id', how='left')
daily_geo_sales = store_sales_geo.groupby(['date', 'region'])['quantity'].sum().reset_index()
daily_geo_sales['date'] = pd.to_datetime(daily_geo_sales['date'])

# 3. Aggregate daily max wind speeds by region to match
store_regions = loc[['store_id', 'region']].drop_duplicates()
regional_weather = weather_by_store_clean.merge(store_regions, on='store_id', how='left')
daily_regional_wind = regional_weather.groupby(['date', 'region'])['windspeed_10m_max'].max().reset_index()

# 4. Combine sales volume and wind severity
plot_df = pd.merge(daily_geo_sales, daily_regional_wind, on=['date', 'region'], how='inner')
plot_df = plot_df[plot_df['date'].between('2024-09-01', '2024-09-30')].sort_values(['region', 'date'])

# 5. Set up plot grid for selected key regions
selected_regions = ['West', 'Central', 'South', 'North']
fig, axes = plt.subplots(len(selected_regions), 1, figsize=(14, 12), sharex=True, sharey=True)

# Global normalization for wind speed to ensure colors are comparable
norm = Normalize(vmin=plot_df['windspeed_10m_max'].min(), vmax=plot_df['windspeed_10m_max'].max())
cmap = plt.get_cmap('YlOrRd')

for i, region in enumerate(selected_regions):
    ax = axes[i]
    region_data = plot_df[plot_df['region'] == region].copy()

    # Prepare dates as numeric representation for line collection plotting
    x = mdates.date2num(region_data['date'])
    y = region_data['quantity'].values
    colors_val = region_data['windspeed_10m_max'].values

    # Split line into individual colored segments
    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    lc = LineCollection(segments, cmap=cmap, norm=norm)
    lc.set_array(colors_val)
    lc.set_linewidth(3)

    ax.add_collection(lc)

    # Plot background points so markers reflect daily points
    ax.scatter(region_data['date'], y, c=colors_val, cmap=cmap, norm=norm, edgecolor='grey', s=35, zorder=3)

    # Intercept annotations
    ax.axvline(pd.Timestamp('2024-09-26'), color='darkred', linestyle=':', alpha=0.7)
    ax.text(pd.Timestamp('2024-09-26'), ax.get_ylim()[0] + (ax.get_ylim()[1]-ax.get_ylim()[0])*0.8,
            'Landfall', color='darkred', fontsize=9, fontweight='semibold')

    ax.set_ylabel(f'{region} Region\nUnits Sold', fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)

# Format date axis
axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=3))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=45, ha='right')

# Create a shared colorbar for wind speed
cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, label='Local Wind Speed (km/h)')

fig.suptitle('Regional Sales Volume Colored by Wind Speed Intensity (September 2024)', fontsize=14, fontweight='bold', y=0.95)
plt.subplots_adjust(right=0.91, hspace=0.15)
plt.show()

In [ ]:
from matplotlib.colors import Normalize

# 1. Align the regional sales and regional weather data
store_sales_geo = pd.merge(analysis_trans, loc[['store_id', 'region']], on='store_id', how='left')
daily_geo_sales = store_sales_geo.groupby(['date', 'region'])['quantity'].sum().reset_index()
daily_geo_sales['date'] = pd.to_datetime(daily_geo_sales['date'])

weather_by_store_clean = weather_by_store.copy()
weather_by_store_clean['date'] = pd.to_datetime(weather_by_store_clean['date'])
store_regions = loc[['store_id', 'region']].drop_duplicates()
regional_weather = weather_by_store_clean.merge(store_regions, on='store_id', how='left')
daily_regional_wind = regional_weather.groupby(['date', 'region'])['windspeed_10m_max'].max().reset_index()

plot_df = pd.merge(daily_geo_sales, daily_regional_wind, on=['date', 'region'], how='inner')
plot_df = plot_df[plot_df['date'].between('2024-09-01', '2024-09-30')].sort_values(['region', 'date'])

# 2. Setup a 2x2 grid of regions (West & Central on Left, South & North on Right)
selected_regions = [['West', 'Central'], ['South', 'North']]
fig = plt.figure(figsize=(18, 12))

# Setup shared normalization for color strips
wind_min, wind_max = plot_df['windspeed_10m_max'].min(), plot_df['windspeed_10m_max'].max()
norm = Normalize(vmin=wind_min, vmax=wind_max)
cmap = plt.get_cmap('YlOrRd')

# GridSpec with 2 columns, each with a sales plot and a weather strip panel
gs = fig.add_gridspec(nrows=4, ncols=2, height_ratios=[4.0, 0.6, 4.0, 0.6], hspace=0.35, wspace=0.18)

axes_sales = []
axes_strips = []

for col_idx, col_regions in enumerate(selected_regions):
    for row_idx, region in enumerate(col_regions):
        # Row positions in the 4-row GridSpec
        sales_row = 0 if row_idx == 0 else 2
        strip_row = 1 if row_idx == 0 else 3

        # 1. Sales Panel
        ax_sales = fig.add_subplot(gs[sales_row, col_idx])
        axes_sales.append(ax_sales)

        # 2. Thin Weather Strip Panel below sales
        ax_strip = fig.add_subplot(gs[strip_row, col_idx], sharex=ax_sales)
        axes_strips.append(ax_strip)

        region_data = plot_df[plot_df['region'] == region].copy().sort_values('date')
        dates = region_data['date'].values
        sales_vals = region_data['quantity'].values
        wind_vals = region_data['windspeed_10m_max'].values

        # Plot sales line graph
        ax_sales.plot(dates, sales_vals, color='#24323E', linewidth=2.5, marker='o', markersize=4, label=f'{region} Sales')
        ax_sales.set_ylabel('Units Sold', fontweight='bold', fontsize=10)
        ax_sales.set_title(f'{region} Region', fontweight='bold', fontsize=12, loc='left', pad=6)
        ax_sales.grid(True, linestyle='--', alpha=0.7)
        ax_sales.set_axisbelow(True)

        ax_sales.axvline(pd.Timestamp('2024-09-26'), color='darkred', linestyle=':', alpha=0.8)
        ax_sales.set_ylim(0, 1100)

        # Hide x-axis labels on sales plot
        plt.setp(ax_sales.get_xticklabels(), visible=False)

        # Plot storm wind intensity strip
        x_mesh = mdates.date2num(dates)
        dx = np.diff(x_mesh)[0] if len(x_mesh) > 1 else 1.0
        edges = np.append(x_mesh - dx/2, x_mesh[-1] + dx/2)

        img = ax_strip.pcolormesh(edges, [0, 1], wind_vals.reshape(1, -1), cmap=cmap, norm=norm, edgecolors='face')
        ax_strip.set_yticks([])
        ax_strip.set_ylabel('Wind Speed', fontsize=9, rotation=0, labelpad=30, va='center', fontweight='bold')
        ax_strip.axvline(pd.Timestamp('2024-09-26'), color='black', linestyle=':', alpha=0.8)

        # Remove borders of color strips
        for spine in ['top', 'bottom', 'left', 'right']:
            ax_strip.spines[spine].set_visible(False)

        # Format date axis specifically on the weather strips
        ax_strip.xaxis.set_major_locator(mdates.DayLocator(interval=3))
        ax_strip.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        plt.setp(ax_strip.get_xticklabels(), rotation=30, ha='right', fontsize=9)

# Add global colorbar for local wind speed intensity
cbar_ax = fig.add_axes([0.94, 0.15, 0.02, 0.7])
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, label='Local Daily Max Wind Speed (km/h)')

fig.suptitle('Wave of Storm Preparedness: Regional Sales Patterns & Storm Track Alignment', fontsize=16, fontweight='bold', y=0.97)
plt.subplots_adjust(right=0.91, top=0.91, bottom=0.08, hspace=0.35, wspace=0.18)
plt.show()

In [ ]:

# 1. Ensure dates are datetimes in our datasets
analysis_trans['date'] = pd.to_datetime(analysis_trans['date'])
weather_by_store_clean = weather_by_store.copy()
weather_by_store_clean['date'] = pd.to_datetime(weather_by_store_clean['date'])

# 2. Get baseline daily sales per region (Sept 1 - Sept 15)
store_sales_geo = pd.merge(analysis_trans, loc[['store_id', 'region']], on='store_id', how='left')
regional_daily_sales = store_sales_geo.groupby(['date', 'region'])['quantity'].sum().reset_index()

region_baselines = (
    regional_daily_sales[regional_daily_sales['date'].between('2024-09-01', '2024-09-15')]
    .groupby('region')['quantity']
    .mean()
    .rename('baseline_sales')
)

# 3. Merge baseline back and calculate daily % change for the entire month
heatmap_df = pd.merge(regional_daily_sales, region_baselines, on='region', how='left')
heatmap_df['sales_change_pct'] = ((heatmap_df['quantity'] - heatmap_df['baseline_sales']) / heatmap_df['baseline_sales']) * 100

# Filter to September
heatmap_df = heatmap_df[heatmap_df['date'].between('2024-09-01', '2024-09-30')]

# 4. Get peak wind date for each region to sort regions by storm arrival
store_regions = loc[['store_id', 'region']].drop_duplicates()
regional_weather = weather_by_store_clean.merge(store_regions, on='store_id', how='left')
regional_peak_wind_dates = (
    regional_weather.groupby(['region', 'date'])['windspeed_10m_max'].max().reset_index()
)
# Find the date of maximum wind speed for each region
region_arrival_sort = (
    regional_peak_wind_dates.loc[regional_peak_wind_dates.groupby('region')['windspeed_10m_max'].idxmax()]
    .sort_values('date', ascending=True) # Earliest arrival first
)

sorted_regions = region_arrival_sort['region'].tolist()

# 5. Pivot data for the heatmap
pivot_df = heatmap_df.pivot(index='region', columns='date', values='sales_change_pct')
pivot_df = pivot_df.reindex(sorted_regions) # Apply the sorted regional order

# Format the columns for cleaner display
pivot_df.columns = [d.strftime('%b %d') for d in pivot_df.columns]

# 6. Plotting the Heatmap with Track Overlay
plt.figure(figsize=(16, 7))

# Center color bar at 0 (white) using a diverging colormap
sns.heatmap(
    pivot_df,
    cmap='RdBu_r',
    center=0,
    robust=True,
    cbar_kws={'label': 'Sales Change vs. Baseline (%)'}
)

# Calculate the positions for the diagonal track line overlaying peak wind speeds
# Let's map each region's peak wind date index on the x-axis to draw a trendline
x_indices = []
for reg in sorted_regions:
    peak_date = region_arrival_sort[region_arrival_sort['region'] == reg]['date'].iloc[0]
    formatted_date_str = peak_date.strftime('%b %d')
    x_idx = pivot_df.columns.get_loc(formatted_date_str)
    x_indices.append(x_idx + 0.5) # Center of the heatmap cell

y_indices = np.arange(len(sorted_regions)) + 0.5

# Plot the diagonal storm track line
plt.plot(x_indices, y_indices, color='black', linestyle='--', linewidth=3, marker='o', markersize=8, label='Storm Arrival (Peak Wind)')

# Visual Formatting
plt.title('Wave of Storm Preparedness: Regional Sales Changes & Storm Path Arrival', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Date (September 2024)', fontweight='bold', labelpad=10)
plt.ylabel('Region (Sorted by Storm Arrival)', fontweight='bold', labelpad=10)
plt.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Align daily regional sales lift and daily regional wind speeds
# Both daily_geo_sales and daily_regional_wind have date and region.
dose_response_df = pd.merge(heatmap_df, daily_regional_wind, on=['date', 'region'], how='inner')

# 2. Plotting the Dose-Response relationship
fig, ax = plt.subplots(figsize=(10, 6))
ax.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
sns.scatterplot(
    data=dose_response_df,
    x='windspeed_10m_max',
    y='sales_change_pct',
    hue='region',
    style='region',
    s=100,
    alpha=0.8
)

# Add a trendline to visualize the overarching dose-response pattern
sns.regplot(
    data=dose_response_df,
    x='windspeed_10m_max',
    y='sales_change_pct',
    scatter=False,
    color='black',
    line_kws={'linestyle': '--', 'linewidth': 1.5, 'label': 'Overarching Trend'}
)

# Visual enhancements
ax.set_title('Dose-Response: Regional Sales Lift vs. Local Wind Speed Severity', fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('Local Max Daily Wind Speed (km/h)', fontweight='bold')
ax.set_ylabel('Sales Volume Lift vs. Baseline (%)', fontweight='bold')
ax.legend(title='Region', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Calculate correlation to quantify the strength of the relationship
corr_val = dose_response_df['windspeed_10m_max'].corr(dose_response_df['sales_change_pct'])
print(f"Correlation between daily local wind speed and regional sales lift: r = {corr_val:.4f}")

The geographi analysis reveals a stark difference in consumer behavior based on proximity to the storm's path. Stores in the direct impact zones (such as coastal or central regions depending on the storm's trajectory) experienced a much sharper and earlier demand surge, peaking roughly 48 to 72 hours before landfall as resident rushed to secure emergency supplied. In contrast, stores in regions further from the projected path saw a milder, flatter increase in sales, reflecting cautionary buying rather than panic buying. Additionally, sales in the hardest-hit--regions dropped to zero on the day of landfall (September 26) due to store closures, while peripheral regions maintained some baseline level of transaction volume.

###Q5: Signal or Artifact - SQL queries on the inventory database, a cross-check against the transactions, and a recommendation

In [ ]:
# Q5: Compare transaction totals with the daily sales database
def parse_db_date(value):
    value = str(value).strip()
    if re.fullmatch(r"\d{8}", value):
        return pd.to_datetime(value, format="%Y%m%d", errors="coerce")
    return pd.to_datetime(value, errors="coerce")

db_sales = daily_sales_db.copy()
db_sales["date"] = db_sales["date"].apply(parse_db_date)
db_sales["store_id"] = db_sales["store_id"].map(clean_code)
db_sales["category"] = (
    db_sales["category"].astype(str).str.strip().str.lower().str.title()
    .replace(category_map)
)

transaction_totals = (
    analysis_trans.groupby(["date", "store_id", "category"], as_index=False)
    .agg(transaction_units=("quantity", "sum"))
)

database_totals = (
    db_sales.groupby(["date", "store_id", "category"], as_index=False)
    .agg(database_units=("total_units", "sum"))
)

reconciliation = transaction_totals.merge(
    database_totals,
    on=["date", "store_id", "category"],
    how="outer",
    indicator=True
)

reconciliation["transaction_units"] = reconciliation["transaction_units"].fillna(0)
reconciliation["database_units"] = reconciliation["database_units"].fillna(0)
reconciliation["unit_difference"] = (
    reconciliation["transaction_units"] - reconciliation["database_units"]
)

print("Rows matched between sources:")
print(reconciliation["_merge"].value_counts())
print("Mean absolute unit difference:",
      round(reconciliation["unit_difference"].abs().mean(), 2))
print("Rows with a nonzero difference:",
      (reconciliation["unit_difference"] != 0).sum())

display(
    reconciliation[reconciliation["unit_difference"] != 0]
    .sort_values("unit_difference", key=abs, ascending=False)
    .head(10)
)

# Check how often inventory was at or below its reorder point
inventory = inv_levels_db.copy()
inventory["date"] = inventory["date"].apply(parse_db_date)
inventory["store_id"] = inventory["store_id"].map(clean_code)
inventory["sku"] = (
    inventory["sku"].astype(str).str.strip().str.upper().replace(sku_map)
)
inventory["below_reorder"] = (
    inventory["units_on_hand"] <= inventory["reorder_point"]
)

inventory["period"] = "Other"
inventory.loc[
    inventory["date"].between(baseline_start, baseline_end), "period"
] = "Baseline"
inventory.loc[
    inventory["date"].between(surge_start, surge_end), "period"
] = "Pre-landfall surge"
inventory = inventory[inventory["period"] != "Other"].copy()

# Map cleaned SKUs to cleaned product categories
catalog_lookup = (
    cat.groupby("sku", as_index=False)
    .agg(category=("category", "first"),
         product_name=("product_name", "first"))
)

inventory = inventory.merge(catalog_lookup, on="sku", how="left")

inventory_risk = (
    inventory.groupby(["period", "category"], as_index=False)
    .agg(below_reorder_rate=("below_reorder", "mean"))
)

display(inventory_risk.sort_values(
    ["category", "period"]
).round(3))

fig, ax = plt.subplots(figsize=(10, 6))
ax.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
sns.barplot(
    data=inventory_risk,
    x="category",
    y="below_reorder_rate",
    hue="period",
    ax=ax
)
ax.set_title("Share of Inventory Records at or Below Reorder Point")
ax.set_xlabel("Product category")
ax.set_ylabel("Share of store-SKU records")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# List products most often at or below reorder point during the surge window
at_risk_skus = (
    inventory[inventory["period"] == "Pre-landfall surge"]
    .groupby(["sku", "product_name"], as_index=False)
    .agg(
        store_days_below_reorder=("below_reorder", "sum"),
        average_units_on_hand=("units_on_hand", "mean")
    )
    .sort_values("store_days_below_reorder", ascending=False)
)

display(at_risk_skus.head(10))

In [ ]:
#Q5: Signal or Artifact
#Is the surge real demand, or an inventory or data effect?
#Added: Query the SQLite inventory database to cross-check transactional demand
#The connection 'conn' was established in the raw data cell
query = """
  SELECT date, store_id, sku, units_on_hand, units_on_order
  FROM inventory_levels
  WHERE date BETWEEN '2024-09-01' AND '2024-09-30'
"""
inventory_data = pd.read_sql(query, conn)
inventory_data['date'] = pd.to_datetime(inventory_data['date'])

#Added: Map SKUs in the inventory table to match out cleaned transaction SKUs
inventory_data['sku'] = inventory_data['sku'].replace(sku_map)

#Added: Focus on the most critical high-volume survival items to cross-check
key_skus = ['WATER24PK', "POPTART-STRAW", 'BAT-AA4']
inv_filtered = inventory_data[inventory_data['sku'].isin(key_skus)]
daily_inv = inv_filtered.groupby('date')[['units_on_hand', 'units_on_order']].sum().reset_index()

#Added: Aggregate actual sales for these same SKUs to compare against inventory
sales_filtered = analysis_trans[analysis_trans['sku'].isin(key_skus)]
daily_sales_skus = sales_filtered.groupby('date')['quantity'].sum().reset_index()

#Added: Visualization comparing Physical Inventory vs Point-of-Sale Demand
fig, ax1 = plt.subplots(figsize=(14,7))
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_axisbelow(True)
#Plot Inventory on primary y-axis
ax1.plot(daily_inv['date'], daily_inv['units_on_hand'], 'g-', linewidth=2, marker='s', label='Units on Hand (Stock)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Total Units on Hand', color='g', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='g')

#Plot Sales on Secondary y-axis
ax2 = ax1.twinx()
ax2.plot(daily_sales_skus['date'], daily_sales_skus['quantity'], 'b--', linewidth=2, marker='o', label = 'Units Sold (Demand)')
ax2.set_ylabel('Total Units Sold', color='b', fontweight='bold')
ax2.tick_params(axis='y', labelcolor = 'b')

#Mark the day of landfall
landfall_date = pd.Timestamp("2024-09-26")
ax1.axvline(landfall_date, color="red", linestyle=":", linewidth=2, label="Landfall: Sept. 26")

plt.title('Signal Check: Real Demand vs Physical Inventory Levels (Top Survival SKUs)')
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.92), ncol=3)
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout()
plt.show()

#Close the database connection
conn.close()

Signal or Artifact
The surge observed in the transactional data is genuine demand signal, not a data artifact or automated inventory effect. By cross-checking the Point-of-Sale (POS) records against SQLite inventory_levels table for key survival SKUs (like bottled water, batteries, and Pop-Tarts), we see an exact inverse relationship. As sales volume spiked dramatically between Sepetember 23 and September 25, the physical units_on_hand plummeted in real-time, verifying that goods were physically leaving the shelves. If this were a data artifact (e.g., duplicated transaction logs), the inventory database would not reflect a corresponding depletion of physical stock.

Final Recommendation to Walmart Leadership:
Based on the validates 48-72 hour demand surge preceding landfall, Walmart should implement a predictive staging strategy for stores in the projected cone of impact. High-fragmentation, high-velocity survival items (water, batteries, non-perishable foods) must be preemptively pushed from distribution centers to store backrooms at least 96 hours (4 days) prior to estimated landfall. Additionally, automated SKU consolidation protocols should be enforced at the POS level to prevent system fragmentation during panic buying, ensuring accurate, real-time inventory triggers that automatically prompt reorders before stockouts occur.